# THzSim2 User Workflow

This notebook is for study and simulation workflows in Google Colab or locally. It keeps the measurement definition first, then the sample and study sweep, then writes a single CSV setup file and runs the study from that file.


## Google Colab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Podrimate/THz_sim_application/blob/main/notebooks/THzSim_User_Workflow.ipynb)

This notebook automatically installs the package from GitHub when it runs inside Colab. If you want to use a measured reference trace, there is also a Colab upload helper below.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
PACKAGE_IMPORT_OK = False
DEFAULT_PIP_SPEC = 'git+https://github.com/Podrimate/THz_sim_application.git'

try:
    import thzsim2  # noqa: F401
    PACKAGE_IMPORT_OK = True
except Exception:
    PACKAGE_IMPORT_OK = False

if not PACKAGE_IMPORT_OK:
    repo_candidates = [Path.cwd(), Path.cwd().parent]
    for candidate in repo_candidates:
        if (candidate / 'thzsim2').exists():
            sys.path.insert(0, str(candidate))
            try:
                import thzsim2  # noqa: F401
                PACKAGE_IMPORT_OK = True
                break
            except Exception:
                pass

if IN_COLAB and not PACKAGE_IMPORT_OK:
    pip_spec = os.environ.get('THZSIM_PIP_SPEC', DEFAULT_PIP_SPEC).strip()
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pip_spec])
    import thzsim2  # noqa: F401
    PACKAGE_IMPORT_OK = True

if not PACKAGE_IMPORT_OK:
    raise RuntimeError(
        'Could not import thzsim2. Open this notebook inside the repo, or in Colab let the install cell pull from GitHub.'
    )


## Setup / Imports

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from thzsim2.notebook_api import (
    ConstantNK,
    Drude,
    DrudeLorentz,
    Fit,
    Layer,
    LorentzOscillator,
    NKFile,
    build_study_setup,
    drude_gamma_thz_from_tau_ps,
    drude_plasma_freq_thz_from_sigma_tau,
    load_study_setup_csv,
    run_study_from_setup_csv,
    show_study_heatmaps,
    write_study_setup_csv,
)

workspace_root = Path.cwd()
if not (workspace_root / 'thzsim2').exists() and (workspace_root.parent / 'thzsim2').exists():
    workspace_root = workspace_root.parent

workflow_root = workspace_root / 'notebooks' / 'workflow_outputs'
materials_root = workflow_root / 'materials'
uploads_root = workflow_root / 'uploads'
workflow_root.mkdir(parents=True, exist_ok=True)
materials_root.mkdir(parents=True, exist_ok=True)
uploads_root.mkdir(parents=True, exist_ok=True)

print('workspace_root =', workspace_root)
print('workflow_root  =', workflow_root)


## Helper Functions

In [ ]:
def show_setup_sections(setup_csv_path):
    setup = load_study_setup_csv(setup_csv_path)
    for key in ('meta', 'reference', 'sample', 'measurement', 'study', 'notes'):
        if key not in setup:
            continue
        print(f'\n[{key}]')
        print(json.dumps(setup[key], indent=2))


def upload_single_csv(target_dir, *, prompt):
    target_dir = Path(target_dir)
    target_dir.mkdir(parents=True, exist_ok=True)
    try:
        from google.colab import files
    except Exception as exc:
        raise RuntimeError('Colab upload is only available inside Google Colab.') from exc

    print(prompt)
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError('No file was uploaded.')
    name, payload = next(iter(uploaded.items()))
    path = target_dir / name
    path.write_bytes(payload)
    print('saved to', path)
    return path


def write_example_nk_files(material_dir):
    material_dir = Path(material_dir)
    material_dir.mkdir(parents=True, exist_ok=True)
    freq = np.linspace(0.2, 3.0, 120)
    si_path = material_dir / 'si_example_thz_nk.csv'
    sio2_path = material_dir / 'sio2_backside_example_thz_nk.csv'
    si_n = 3.42 + 0.02 * np.exp(-((freq - 1.8) / 0.9) ** 2)
    si_k = 0.002 + 0.001 * np.exp(-((freq - 2.4) / 0.5) ** 2)
    sio2_n = 1.95 + 0.08 * np.exp(-((freq - 1.2) / 0.35) ** 2)
    sio2_k = 0.18 + 0.12 * np.exp(-((freq - 1.2) / 0.25) ** 2)
    header = 'freq_thz,n,k\n'
    si_rows = '\n'.join(f'{f:.8g},{n:.8g},{k:.8g}' for f, n, k in zip(freq, si_n, si_k))
    sio2_rows = '\n'.join(f'{f:.8g},{n:.8g},{k:.8g}' for f, n, k in zip(freq, sio2_n, sio2_k))
    si_path.write_text(header + si_rows + '\n', encoding='utf-8')
    sio2_path.write_text(header + sio2_rows + '\n', encoding='utf-8')
    return si_path, sio2_path


def preview_study_outputs(study_result, *, label=None):
    if label is not None:
        print(f'\n=== {label} ===')
    print('out dir      :', study_result.out_dir)
    print('summary csv   :', study_result.summary_csv_path)
    print('case count    :', len(study_result.summary_rows))
    print('first keys    :', list(study_result.summary_rows[0].keys())[:18])
    show_study_heatmaps(study_result, contains='normalized-mse__', max_images=4)


## 1. Measurement And Reference Setup

Choose transmission or reflection, angle of incidence, and whether the reference is generated or uploaded/measured.

In [ ]:
use_measured_reference = False
reference_csv_path = None
measurement_mode = 'transmission'  # 'transmission' or 'reflection'
incident_angle_deg = 25.0
polarization = 'p'  # 's', 'p', or 'mixed'
polarization_mix = 0.5  # only used when polarization='mixed'


## 1a. Acquire The Measured Reference CSV If Needed

In [ ]:
if use_measured_reference:
    if reference_csv_path is None:
        if IN_COLAB:
            reference_csv_path = upload_single_csv(uploads_root, prompt='Upload the measured reference CSV file.')
        else:
            raise RuntimeError('Set reference_csv_path to a local CSV path when not using Colab uploads.')
    reference_setup = {
        'kind': 'measured_csv',
        'path': reference_csv_path,
        'time_column': None,
        'signal_column': None,
        'prepare': {
            'output_root': workflow_root / 'runs',
            'run_label': 'user-workflow-measured-reference',
            'noise': None,
        },
    }
else:
    reference_setup = {
        'kind': 'generated_pulse',
        'generate': {
            'model': 'sech_carrier',
            'sample_count': 2048,
            'dt_ps': 0.02,
            'time_center_ps': 12.0,
            'pulse_center_ps': 6.5,
            'tau_ps': 0.28,
            'f0_thz': 0.9,
            'amp': 1.0,
            'phi_rad': 0.0,
            'pad_factor': 1,
        },
        'prepare': {
            'output_root': workflow_root / 'runs',
            'run_label': 'user-workflow-generated-reference',
            'noise': None,
        },
    }

measurement_setup = {
    'mode': measurement_mode,
    'angle_deg': incident_angle_deg,
    'polarization': polarization,
    'polarization_mix': polarization_mix if polarization == 'mixed' else None,
    'reference_standard': {'kind': 'identity'},
}
print(json.dumps(measurement_setup, indent=2))


## 2. Arbitrary Sample And Study Sweep

In [ ]:
sample_layers = [
    Layer(
        name='coating',
        thickness_um=Fit(120.0, abs_min=70.0, abs_max=170.0, label='coating_thickness_um'),
        material=ConstantNK(
            n=Fit(2.15, abs_min=1.8, abs_max=2.5, label='coating_n'),
            k=Fit(0.03, abs_min=0.0, abs_max=0.08, label='coating_k'),
        ),
    ),
    Layer(
        name='substrate',
        thickness_um=500.0,
        material=ConstantNK(n=1.55, k=0.0),
    ),
]

study_config = {
    'truth': {
        'layers[0].thickness_um': [90.0, 120.0, 150.0],
        'layers[0].material.n': [2.0, 2.15, 2.3],
        'layers[0].material.k': [0.01, 0.03, 0.05],
    },
    'noise_dynamic_range_db': [55.0, 70.0],
    'replicates': 1,
    'seed': 123,
    'metric': 'mse',
    'optimizer': {
        'method': 'L-BFGS-B',
        'options': {'maxiter': 20},
        'global_options': {'maxiter': 2, 'popsize': 5, 'seed': 123},
        'fd_rel_step': 1e-4,
    },
    'out_dir': workflow_root / 'study_outputs' / 'main_study',
}

sample_options = {
    'n_in': 1.0,
    'n_out': 1.0,
    'overlay_imported': True,
    'sample_out_dir': workflow_root / 'sample_builds' / 'main_sample',
}


## 3. Save Everything Into One CSV Setup File

In [ ]:
setup = build_study_setup(
    reference=reference_setup,
    layers=sample_layers,
    measurement=measurement_setup,
    study=study_config,
    n_in=sample_options['n_in'],
    n_out=sample_options['n_out'],
    overlay_imported=sample_options['overlay_imported'],
    sample_out_dir=sample_options['sample_out_dir'],
    notes='Main user workflow setup',
)

setup_csv_path = workflow_root / 'study_setup_main.csv'
write_study_setup_csv(setup_csv_path, setup)
print('setup_csv_path =', setup_csv_path)
show_setup_sections(setup_csv_path)


## 4. Run The Study From The CSV File Only

In [ ]:
study_result = run_study_from_setup_csv(setup_csv_path)
preview_study_outputs(study_result, label='main_workflow')


_ = ## 5. Review The Generated Outputs

In [ ]:
summary_rows = study_result.summary_rows
for row in summary_rows[:3]:
    print({
        'case_id': row['case_id'],
        'normalized_mse': row['normalized_mse'],
        'relative_l2': row['relative_l2'],
        'true__coating_thickness_um': row.get('true__coating_thickness_um'),
        'fit__coating_thickness_um': row.get('fit__coating_thickness_um'),
    })


In [ ]:
_heatmap = show_study_heatmaps(study_result, contains='normalized-mse__', max_images=6)


In [ ]:
_heatmap = show_study_heatmaps(study_result, contains='relative-l2__', max_images=6)


In [ ]:
_heatmap = show_study_heatmaps(study_result, contains='signed-err', max_images=9)


## Optional Scenario Templates

In [ ]:
def make_single_layer_drude_template(output_root):
    gamma_grid = [drude_gamma_thz_from_tau_ps(tau) for tau in (0.2, 0.5, 1.0)]
    plasma_grid = [drude_plasma_freq_thz_from_sigma_tau(sig, 0.5) for sig in (2e3, 5e3, 1e4)]
    layers = [
        Layer(
            name='drude_layer',
            thickness_um=Fit(60.0, abs_min=20.0, abs_max=120.0, label='drude_thickness_um'),
            material=Drude(
                eps_inf=12.0,
                plasma_freq_thz=Fit(0.4, abs_min=0.1, abs_max=1.2, label='drude_plasma_freq_thz'),
                gamma_thz=Fit(0.4, abs_min=0.05, abs_max=2.0, label='drude_gamma_thz'),
            ),
        )
    ]
    study = {
        'truth': {
            'layers[0].thickness_um': [30.0, 60.0, 90.0],
            'layers[0].material.plasma_freq_thz': plasma_grid,
            'layers[0].material.gamma_thz': gamma_grid,
        },
        'noise_dynamic_range_db': [55.0, 70.0],
        'replicates': 1,
        'seed': 123,
        'optimizer': {
            'method': 'L-BFGS-B',
            'options': {'maxiter': 20},
            'global_options': {'maxiter': 2, 'popsize': 5, 'seed': 123},
            'fd_rel_step': 1e-4,
        },
        'out_dir': Path(output_root) / 'drude_template_study',
    }
    measurement = {'mode': 'transmission', 'angle_deg': 0.0, 'polarization': 's', 'reference_standard': {'kind': 'identity'}}
    return layers, measurement, study


def make_two_layer_lorentz_si_template(output_root, material_dir):
    si_path, _ = write_example_nk_files(material_dir)
    layers = [
        Layer(
            name='lorentz_film',
            thickness_um=Fit(30.0, abs_min=10.0, abs_max=60.0, label='film_thickness_um'),
            material=DrudeLorentz(
                eps_inf=Fit(2.2, abs_min=1.5, abs_max=3.5, label='film_eps_inf'),
                plasma_freq_thz=0.0,
                gamma_thz=0.0,
                oscillators=(
                    LorentzOscillator(
                        delta_eps=Fit(2.8, abs_min=1.0, abs_max=5.0, label='osc1_delta_eps'),
                        resonance_thz=Fit(0.8, abs_min=0.4, abs_max=1.3, label='osc1_resonance_thz'),
                        gamma_thz=Fit(0.15, abs_min=0.05, abs_max=0.5, label='osc1_gamma_thz'),
                    ),
                    LorentzOscillator(
                        delta_eps=Fit(1.6, abs_min=0.5, abs_max=4.0, label='osc2_delta_eps'),
                        resonance_thz=Fit(1.8, abs_min=1.1, abs_max=2.5, label='osc2_resonance_thz'),
                        gamma_thz=Fit(0.22, abs_min=0.08, abs_max=0.7, label='osc2_gamma_thz'),
                    ),
                ),
            ),
        ),
        Layer(
            name='si_substrate',
            thickness_um=Fit(450.0, abs_min=300.0, abs_max=700.0, label='si_thickness_um'),
            material=NKFile(si_path),
        ),
    ]
    study = {
        'truth': {
            'layers[0].thickness_um': [20.0, 30.0],
            'layers[0].material.eps_inf': [2.0, 2.4],
            'layers[0].material.oscillators[0].delta_eps': [2.4, 3.0],
            'layers[0].material.oscillators[0].resonance_thz': [0.75, 0.9],
            'layers[0].material.oscillators[1].delta_eps': [1.2, 1.8],
            'layers[0].material.oscillators[1].resonance_thz': [1.6, 1.95],
            'layers[1].thickness_um': [420.0, 520.0],
        },
        'noise_dynamic_range_db': [60.0, 75.0],
        'replicates': 1,
        'seed': 321,
        'optimizer': {
            'method': 'L-BFGS-B',
            'options': {'maxiter': 25},
            'global_options': {'maxiter': 2, 'popsize': 6, 'seed': 321},
            'fd_rel_step': 1e-4,
        },
        'out_dir': Path(output_root) / 'lorentz_si_template_study',
    }
    measurement = {'mode': 'transmission', 'angle_deg': 15.0, 'polarization': 's', 'reference_standard': {'kind': 'identity'}}
    return layers, measurement, study


def make_three_layer_backside_template(output_root, material_dir):
    si_path, sio2_path = write_example_nk_files(material_dir)
    layers = [
        Layer(
            name='lorentz_film',
            thickness_um=Fit(28.0, abs_min=10.0, abs_max=55.0, label='film_thickness_um'),
            material=DrudeLorentz(
                eps_inf=Fit(2.4, abs_min=1.5, abs_max=3.5, label='film_eps_inf'),
                plasma_freq_thz=0.0,
                gamma_thz=0.0,
                oscillators=(
                    LorentzOscillator(
                        delta_eps=Fit(2.6, abs_min=1.0, abs_max=5.0, label='osc1_delta_eps'),
                        resonance_thz=Fit(0.85, abs_min=0.4, abs_max=1.3, label='osc1_resonance_thz'),
                        gamma_thz=Fit(0.16, abs_min=0.05, abs_max=0.5, label='osc1_gamma_thz'),
                    ),
                    LorentzOscillator(
                        delta_eps=Fit(1.4, abs_min=0.5, abs_max=4.0, label='osc2_delta_eps'),
                        resonance_thz=Fit(1.9, abs_min=1.1, abs_max=2.6, label='osc2_resonance_thz'),
                        gamma_thz=Fit(0.24, abs_min=0.08, abs_max=0.7, label='osc2_gamma_thz'),
                    ),
                ),
            ),
        ),
        Layer(
            name='si_substrate',
            thickness_um=Fit(500.0, abs_min=350.0, abs_max=750.0, label='si_thickness_um'),
            material=NKFile(si_path),
        ),
        Layer(
            name='sio2_backside',
            thickness_um=50000.0,
            material=NKFile(sio2_path),
        ),
    ]
    study = {
        'truth': {
            'layers[0].thickness_um': [22.0, 32.0],
            'layers[0].material.eps_inf': [2.0, 2.6],
            'layers[0].material.oscillators[0].delta_eps': [2.2, 2.8],
            'layers[0].material.oscillators[1].resonance_thz': [1.7, 2.05],
            'layers[1].thickness_um': [450.0, 550.0],
        },
        'noise_dynamic_range_db': [60.0, 75.0],
        'replicates': 1,
        'seed': 456,
        'optimizer': {
            'method': 'L-BFGS-B',
            'options': {'maxiter': 25},
            'global_options': {'maxiter': 2, 'popsize': 6, 'seed': 456},
            'fd_rel_step': 1e-4,
        },
        'out_dir': Path(output_root) / 'lorentz_si_sio2_template_study',
    }
    measurement = {'mode': 'transmission', 'angle_deg': 20.0, 'polarization': 'p', 'reference_standard': {'kind': 'identity'}}
    return layers, measurement, study


## Run Optional Scenario Templates

Choose one or more scenario names below. The runner will build a fresh setup CSV for each selected scenario, execute the study, and collect the results in `scenario_results`.

In [ ]:
available_scenarios = {
    'main_workflow': lambda: {
        'reference': reference_setup,
        'layers': sample_layers,
        'measurement': measurement_setup,
        'study': study_config,
        'n_in': sample_options['n_in'],
        'n_out': sample_options['n_out'],
        'overlay_imported': sample_options['overlay_imported'],
        'sample_out_dir': sample_options['sample_out_dir'],
        'setup_csv_path': workflow_root / 'study_setup_main.csv',
    },
    'single_layer_drude_template': lambda: (lambda layers, measurement, study: {
        'reference': reference_setup,
        'layers': layers,
        'measurement': measurement,
        'study': study,
        'n_in': 1.0,
        'n_out': 1.0,
        'overlay_imported': True,
        'sample_out_dir': workflow_root / 'sample_builds' / 'single_layer_drude_template',
        'setup_csv_path': workflow_root / 'study_setup__single_layer_drude_template.csv',
    })(*make_single_layer_drude_template(workflow_root / 'study_outputs')),
    'two_layer_lorentz_si_template': lambda: (lambda layers, measurement, study: {
        'reference': reference_setup,
        'layers': layers,
        'measurement': measurement,
        'study': study,
        'n_in': 1.0,
        'n_out': 1.0,
        'overlay_imported': True,
        'sample_out_dir': workflow_root / 'sample_builds' / 'two_layer_lorentz_si_template',
        'setup_csv_path': workflow_root / 'study_setup__two_layer_lorentz_si_template.csv',
    })(*make_two_layer_lorentz_si_template(workflow_root / 'study_outputs', materials_root)),
    'three_layer_backside_template': lambda: (lambda layers, measurement, study: {
        'reference': reference_setup,
        'layers': layers,
        'measurement': measurement,
        'study': study,
        'n_in': 1.0,
        'n_out': 1.0,
        'overlay_imported': True,
        'sample_out_dir': workflow_root / 'sample_builds' / 'three_layer_backside_template',
        'setup_csv_path': workflow_root / 'study_setup__three_layer_backside_template.csv',
    })(*make_three_layer_backside_template(workflow_root / 'study_outputs', materials_root)),
}

selected_scenarios = []
print('available scenarios:', list(available_scenarios.keys()))
print('selected scenarios :', selected_scenarios)


In [ ]:
scenario_results = {}
for scenario_name in selected_scenarios:
    scenario = available_scenarios[scenario_name]()
    setup = build_study_setup(
        reference=scenario['reference'],
        layers=scenario['layers'],
        measurement=scenario['measurement'],
        study=scenario['study'],
        n_in=scenario['n_in'],
        n_out=scenario['n_out'],
        overlay_imported=scenario['overlay_imported'],
        sample_out_dir=scenario['sample_out_dir'],
        notes=f'Optional scenario: {scenario_name}',
    )
    setup_csv = scenario['setup_csv_path']
    write_study_setup_csv(setup_csv, setup)
    result = run_study_from_setup_csv(setup_csv)
    scenario_results[scenario_name] = result
    preview_study_outputs(result, label=scenario_name)
print('scenario_results keys =', list(scenario_results.keys()))


## Sharing This Notebook In Colab

The public Colab link is:

`https://colab.research.google.com/github/Podrimate/THz_sim_application/blob/main/notebooks/THzSim_User_Workflow.ipynb`

If you want to use measured references, upload the CSV from your computer when `use_measured_reference=True`.